# Peak shaving with storage

This notebook uses the **same feeder and 24-hour load pattern** as `base_case.ipynb`, but adds a **storage** controlled to **reduce the active power import at the feeder head** when demand is high (**peak shaving**). Lower feederhead P in those hours also **eases the upstream path** (substation and transmission) that must deliver the energy.

A storage does **not** create energy from nothing: it **moves** it. Power **discharged** at peak must be matched, over the day, by **charging** when there is room to withdraw less from the grid. The case below shows **feederhead P without storage first**, then **adds a Storage element and a StorageController** to illustrate how storage can work as a non-wires alternative.


In [1]:
!pip install py-dss-toolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.1 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/PauloRadatz/opendss-python-examples

Cloning into 'opendss-python-examples'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 88 (delta 27), reused 49 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 313.89 KiB | 3.45 MiB/s, done.
Resolving deltas: 100% (27/27), done.


In [36]:
dss_file_path = "/content/opendss-python-examples/feeder_models/EPRITestCircuits/ckt5/Master_ckt5.dss"

In [18]:
import os
import pathlib
import pandas as pd
import numpy as np
import plotly.graph_objects as go

import py_dss_interface
from py_dss_toolkit import dss_tools

In [5]:
dss = py_dss_interface.DSS()
dss_tools.update_dss(dss)
dss.started

True

In [37]:
dss.text(f"compile [{dss_file_path}]")
dss.text("New Loadshape.load_curve npts=24 interval=1 mult=(0.39204545 0.28977273 0.25568182 0.23863636 0.3125 0.48295455 0.57954545 0.45454545 0.51136364 0.51704545 0.58522727 0.59090909 0.63068182 0.55681818 0.53409091 0.53409091 0.58522727 0.72159091 0.86363636 0.90340909 1.0 0.85795455 0.73863636 0.51136364)")
dss.text("batchedit load..* daily=load_curve")
dss_tools.model.add_line_in_vsource(add_meter=True, add_monitors=True)


In [38]:
dss.text("set mode=daily")
dss.text("solve")

''

## Feederhead power (no storage yet)

The monitor on the line at the source gives **active power (P) through the feederhead** for each **hour of the first daily solve** (1-hour steps). This curve is the **baseline** import before the storage. The next code block estimates how much of that profile **exceeds a notional kW “cap”** (rough scoping) before the full `Storage` model is added.


In [39]:
dss.monitors.first()
feeder_head_powers = dss_tools.results.monitor(dss.monitors.name)
feeder_head_powers["Total P (kW)"] = feeder_head_powers[[" P1 (kW)", " P2 (kW)", " P3 (kW)"]].sum(axis=1)
feeder_head_powers.head()

,Hour,sec,P1 (kW),Q1 (kvar),P2 (kW),Q2 (kvar),P3 (kW),Q3 (kvar),Total P (kW)
0,1.0,0.0,995.105469,446.580292,977.759705,489.169037,949.549683,452.852814,2922.414856
1,2.0,0.0,740.999512,334.274353,728.489319,365.732819,707.500549,339.169403,2176.989380
2,3.0,0.0,656.083313,297.135468,645.116150,324.940491,626.519897,301.540100,1927.719360
3,4.0,0.0,613.581055,278.676697,603.370056,304.657043,585.975952,282.823883,1802.927063
4,5.0,0.0,797.565796,359.155029,784.009399,393.055786,761.428650,364.365204,2343.003845


In [40]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=feeder_head_powers['Hour'], y=feeder_head_powers['Total P (kW)'], mode='lines', name='Total P (kW)'))
fig.update_layout(title='Total P (kW) vs. Hour', xaxis_title='Hour', yaxis_title='Total P (kW)')
fig.show()

In [41]:
peak_shave_target_kw = 6000
efficiency = 0.9

power_above_target = feeder_head_powers[feeder_head_powers['Total P (kW)'] > peak_shave_target_kw]['Total P (kW)'] - peak_shave_target_kw

if not power_above_target.empty:
    # Minimum kW capacity needed for peak shaving
    min_kw_capacity = power_above_target.max()

    # Minimum kWh capacity needed for peak shaving (energy to shave)
    # Assuming each hour represents 1 hour interval
    min_kwh_capacity = power_above_target.sum()

    # Adjust kWh capacity for efficiency if discharging (assuming we are discharging to shave peak)
    min_kwh_capacity_adjusted = min_kwh_capacity / efficiency

    print(f"Peak Shave Target: {peak_shave_target_kw} kW")
    print(f"Minimum kW capacity needed: {min_kw_capacity:.2f} kW")
    print(f"Minimum kWh capacity needed (energy to shave, adjusted for {efficiency*100:.0f}% efficiency): {min_kwh_capacity_adjusted:.2f} kWh")
else:
    print(f"No power values exceed the peak shave target of {peak_shave_target_kw} kW.")
    min_kw_capacity = 0
    min_kwh_capacity_adjusted = 0

Peak Shave Target: 6000 kW
Minimum kW capacity needed: 1281.32 kW
Minimum kWh capacity needed (energy to shave, adjusted for 90% efficiency): 2732.76 kWh


In [91]:
# User-defined storage parameters
storage_kw_capacity = 1500
storage_kwh_capacity = 3000
min_soc_percentage = 0.10 # 10% energy always to be kept
efficiency = 0.9 # New efficiency parameter for charge/discharge

# Calculate the ideal energy that needs to be stored in the storage (from min SOC to full)
net_kwh_to_charge_ideal = storage_kwh_capacity - (storage_kwh_capacity * min_soc_percentage)

# Adjust for charging efficiency: energy drawn from grid = energy stored / efficiency
net_kwh_to_charge = net_kwh_to_charge_ideal / efficiency

print(f"Storage kW Capacity: {storage_kw_capacity} kW")
print(f"Storage kWh Capacity: {storage_kwh_capacity} kWh")
print(f"Minimum SOC to maintain: {min_soc_percentage*100:.0f}% ({storage_kwh_capacity * min_soc_percentage:.0f} kWh)")
print(f"Efficiency (Charge/Discharge): {efficiency*100:.0f}%")
print(f"Maximum energy required to charge (from min SOC to full, considering efficiency): {net_kwh_to_charge:.2f} kWh")
print(f"Peak Shave Target: {peak_shave_target_kw} kW")

# Create a temporary DataFrame for calculation
charging_analysis_df = feeder_head_powers.copy()

# Calculate available power for charging (if Total P is below target)
# If Total P is above target, Available_Charging_Power will be negative, meaning no charging is possible.
charging_analysis_df['Available_Charging_Power'] = peak_shave_target_kw - charging_analysis_df['Total P (kW)']

# Filter for hours where charging is possible (Available_Charging_Power > 0)
# and limit the charging power by the storage's kW capacity (this is the power drawn from the grid)
charging_analysis_df['Actual_Charging_Power_kW'] = charging_analysis_df.apply(
    lambda row: min(row['Available_Charging_Power'], storage_kw_capacity) if row['Available_Charging_Power'] > 0 else 0,
    axis=1
)

# Initialize variables to track charging process
current_charged_kwh = 0 # This will track the total energy *supplied from the grid*
charging_start_hour = None
charging_end_hour = None
charging_possible = False

# Iterate through the DataFrame to find a continuous charging interval
for index, row in charging_analysis_df.iterrows():
    hour = row['Hour']
    actual_charge_power_this_hour = row['Actual_Charging_Power_kW']

    if actual_charge_power_this_hour > 0: # Charging is possible in this hour (power drawn from grid)
        if charging_start_hour is None:
            charging_start_hour = hour
        current_charged_kwh += actual_charge_power_this_hour * 1 # Accumulate energy *supplied from grid*

        if current_charged_kwh >= net_kwh_to_charge: # Check if enough energy *supplied from grid* has been accumulated
            charging_end_hour = hour
            charging_possible = True
            break # Found the interval, exit loop
    else: # If charging is not possible, reset the accumulated energy and start hour for a new continuous interval
        current_charged_kwh = 0
        charging_start_hour = None

# Output the result
if charging_possible:
    print(f"\nThe required {net_kwh_to_charge:.2f} kWh (energy supplied from grid) can be charged within the interval from Hour {charging_start_hour:.0f} to Hour {charging_end_hour:.0f}.")
else:
    print(f"\nCould not find a continuous interval to charge the required {net_kwh_to_charge:.2f} kWh (energy supplied from grid) with the given {storage_kw_capacity} kW storage capacity and {efficiency*100:.0f}% efficiency.")

Storage kW Capacity: 1500 kW
Storage kWh Capacity: 3000 kWh
Minimum SOC to maintain: 10% (300 kWh)
Efficiency (Charge/Discharge): 90%
Maximum energy required to charge (from min SOC to full, considering efficiency): 3000.00 kWh
Peak Shave Target: 6000 kW

The required 3000.00 kWh (energy supplied from grid) can be charged within the interval from Hour 1 to Hour 2.


## Adding storage: peak shave and recharge

The `Storage` and `StorageController` elements are defined with **kW**, **kWh**, and an efficiency curve. The controller is set to **shave** power **above a target (kW)** at the monitored line and to **recharge in a time window** so the storage can absorb energy in lower-import periods.

That is the **“discharge when tight, charge when there is headroom”** idea: the storage must **recharge** after it **discharges** if you want a repeatable daily strategy.


In [139]:
dss.text(f"compile [{dss_file_path}]")
dss.text("New Loadshape.load_curve npts=24 interval=1 mult=(0.39204545 0.28977273 0.25568182 0.23863636 0.3125 0.48295455 0.57954545 0.45454545 0.51136364 0.51704545 0.58522727 0.59090909 0.63068182 0.55681818 0.53409091 0.53409091 0.58522727 0.72159091 0.86363636 0.90340909 1.0 0.85795455 0.73863636 0.51136364)")
dss.text("batchedit load..* daily=load_curve")
dss_tools.model.add_line_in_vsource(add_meter=True, add_monitors=True)

In [140]:
st_location = '796'
dss.text("New XYCurve.Eff npts=4 xarray=[.1  .2  .4  1.0]  yarray=[.86  .9  .93  .97] ")
dss.text(f"New Storage.St phases=3 bus1={st_location} kv=12.47 pf=1 kWrated=1500 %reserve=10 effcurve=Eff kWhrated=3000 %stored=100")
dss.text("new Storagecontroller.sc element=Line.MDV201_CONNECTOR terminal=1 modedis=peakshave modecharge=time kwtarget=6000 Timecharge=1 MonPhase=AVG %ratecharge=50")
dss.text("New Monitor.Mon_St_Powers element=Storage.St mode=1 ppolar=No")

''

## OpenDSS run length: 72 one-hour steps (three days)

`set number=72` with the default **1 h** step in `daily` mode replays the **same 24-hour** load and control **three full times** in series (**72 h** on the time axis). The goal is to show **state of charge and cycling** more clearly.


In [141]:
dss.text("set mode=daily")
dss.text("set number=72")
dss.text("solve")

''

In [142]:
bus_list = [dss_tools.interactive_view.circuit_get_bus_marker(name=st_location, color="red", size=20)]
dss_tools.interactive_view.circuit_plot(bus_markers=bus_list)

In [134]:
dss.monitors.first()
feeder_head_powers = dss_tools.results.monitor(dss.monitors.name)
feeder_head_powers["Total P (kW)"] = feeder_head_powers[[" P1 (kW)", " P2 (kW)", " P3 (kW)"]].sum(axis=1)
feeder_head_powers.head()
fig = go.Figure()
fig.add_trace(go.Scatter(x=feeder_head_powers['Hour'], y=feeder_head_powers['Total P (kW)'], mode='lines', name='Total P (kW)'))
fig.update_layout(title='Total P (kW) vs. Hour', xaxis_title='Hour', yaxis_title='Total P (kW)')
fig.show()

In [135]:
dss_tools.interactive_view.p_vs_time(name="monitor_feeder_head_pq", title="Active Power vs Time at Feeder Head")

In [137]:
storage_powers = dss_tools.results.monitor("Mon_St_Powers")[[" P1 (kW)", " P2 (kW)", " P3 (kW)"]].sum(axis=1)
fig = go.Figure()
fig.add_trace(go.Scatter(x=storage_powers.index, y=storage_powers.values, mode='lines', name='Storage P (kW)'))
fig.update_layout(title='Storage P (kW) vs. Hour', xaxis_title='Hour', yaxis_title='Storage P (kW)')
fig.show()

## How to read the last time-series

- **Feederhead P (with storage)**: compare it to the **earlier (no-storage)** case. The goal is a **lower peak** import in the hours the **peak-shave** logic acts on.
- **Storage P (from the element monitor)**: the trace shows when the device **injects (discharge) (negative in the plot)** to support the bus and when it **absorbs (charge) (positive in the plot)**. The important part is the **complement** between “supporting the grid” and “refilling” over the multi-day run.


## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [pauloradatz.me](https://www.pauloradatz.me)
- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)